## 📁 Key `os` Functions for Filesystem & Paths

| Function | Description | Example | Notes |
|----------|-------------|---------|-------|
| `os.getcwd()` | Returns the **current working directory** (CWD) — the folder where your Python script or terminal session is running from. | `/Users/joelteo/my-project` | Equivalent to `pwd` in terminal. This is the default base for all relative paths like `"."` |
| `os.chdir(path)` | Changes the current working directory to the given `path`. | `os.chdir('data')` | If you do `os.chdir('data')`, then `os.getcwd()` will return `/Users/joelteo/my-project/data`. |
| `os.listdir(path='.')` | Lists all files and folders **inside** the specified directory. `"."` means **current directory**. | `os.listdir('.')` | Use `os.listdir('outputs')` to list another folder. `".."` means parent directory. |
| `os.path.join(*args)` | Joins multiple path parts safely using the correct OS-specific separator (`/` on macOS/Linux, `\` on Windows). | `os.path.join('data', 'train.csv')` → `data/train.csv` | Avoids hardcoding slashes. You can join as many parts as you like: `os.path.join(base, sub, file)`. |
| `os.makedirs(path, exist_ok=True)` | Recursively creates a folder (and any parent folders) if they don’t already exist. | `os.makedirs('outputs/images')` | Similar to `mkdir -p`. If `exist_ok=False` and the folder exists, it throws an error. |
| `os.path.exists(path)` | Checks if a file or folder exists at the given path. | `os.path.exists('data/train.csv')` → `True` | Useful for safe file operations: check before read/write/delete. |
| `os.remove(path)` | Deletes the file at the specified path. | `os.remove('outputs/model.pkl')` | Will throw an error if the file doesn't exist — wrap in `if os.path.exists(...)`. |
| `os.path.dirname(path)` | Returns the directory portion of a path string. | `os.path.dirname('/a/b/c.py')` → `/a/b` | Useful when you want to split a full path into folder vs filename. |
| `os.path.basename(path)` | Returns the file name (last part) of a path. | `os.path.basename('/a/b/c.py')` → `c.py` | Use with `dirname` to separate folder and filename. |
| `os.path.abspath(path)` | Converts a relative path into an absolute path. | `os.path.abspath('outputs/images')` → `/Users/joelteo/my-project/outputs/images` | Makes file references robust when passing to other scripts or APIs. |
| `os.path.relpath(path)` | Gets the **relative path** from the current working directory to a given file or folder. | `os.path.relpath('/Users/joelteo/my-project/src/train.py')` → `src/train.py` | Great for printing short paths or building CLI logs. |

---

### 🔍 Special Path Symbols

| Symbol | Meaning |
|--------|---------|
| `"."`  | **Current directory** (where your script is running or terminal is pointed to) |
| `".."` | **Parent directory** of the current one |

---

### ✅ Practical Example

```python
import os

# Imagine you're in: /Users/joelteo/my-project/
print("CWD:", os.getcwd())  # ➜ /Users/joelteo/my-project

# Safely create subfolders
os.makedirs('outputs/images', exist_ok=True)

# Create relative and absolute paths
img_path = os.path.join('outputs', 'images', 'plot.png')
abs_img_path = os.path.abspath(img_path)
rel_path_from_root = os.path.relpath(img_path)

# Remove the file if it exists
if os.path.exists(img_path):
    os.remove(img_path)

# Break the path into components
dir_name = os.path.dirname(img_path)       # ➜ outputs/images
file_name = os.path.basename(img_path)     # ➜ plot.png

print(f"Relative path: {rel_path_from_root}")
print(f"Absolute path: {abs_img_path}")
print(f"Folder: {dir_name}, File: {file_name}")


# 📦 Optimal Project Structure for SageMaker ML Projects

This is the recommended directory structure when developing **machine learning pipelines with SageMaker**, using **local VS Code** as your development environment.

It ensures:
- Clean separation of logic
- Easy migration to cloud training
- Robust script organization and deployment readiness

---

## 📁 Folder Layout (Local VS Code)

```text
my-project/
├── data/                 ← optional local data for testing and prototyping
├── src/                  ← source code package (used as source_dir)
│   ├── train.py          ← entry point script for SageMaker training
│   ├── utils.py          ← helper functions (e.g., data cleaning, metrics)
│   └── model.py          ← model definition and ML logic
├── notebooks/            ← Jupyter notebooks for EDA and experimentation
├── output/               ← local logs, test results, and outputs
├── run_train.py          ← script to launch SageMaker Estimator and job
├── requirements.txt      ← Python dependencies (optional, for reproducibility)
└── README.md             ← documentation for the project


## Sample src/train.py 
This is the entry point SageMaker runs inside the container. It 
- Loads training and validation data
- Trains your model
- Saves model artifacts to SM_MODEL_DIR
- Writes outputs (metrics, predictions, etc.) to SM_OUTPUT_DATA_DIR

```python
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import joblib

train_dir = os.environ.get('SM_CHANNEL_TRAIN')
val_dir = os.environ.get('SM_CHANNEL_VALIDATION')
model_dir = os.environ.get('SM_MODEL_DIR')

df_train = pd.read_csv(os.path.join(train_dir, 'train.csv'))
df_val = pd.read_csv(os.path.join(val_dir, 'val.csv'))

model = RandomForestClassifier()
model.fit(df_train.drop('target', axis=1), df_train['target'])

joblib.dump(model, os.path.join(model_dir, 'model.joblib'))


## sample run_train.py (project root)

This script is run locally (e.g. via python run_train.py), and launches the SageMaker training job using the Estimator API. It 
- Defines training job configuration
- Zips up the src/ folder
- Uploads code + data paths
- Triggers the SageMaker training job in the cloud

```python
import os
from sagemaker.sklearn.estimator import SKLearn
from sagemaker import get_execution_role

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
SRC_PATH = os.path.join(BASE_DIR, 'src')

estimator = SKLearn(
    entry_point='train.py',
    source_dir=SRC_PATH,
    role=get_execution_role(),
    instance_type='ml.m5.large',
    framework_version='1.2-1',
    base_job_name='my-ml-job'
)

estimator.fit({
    'train': 's3://my-bucket/data/train.csv',
    'validation': 's3://my-bucket/data/val.csv'
})


### What Actually Happens When You Call .fit(...)
Step-by-step Workflow
1. Estimator Initialization
- You create a SKLearn estimator with entry_point='train.py' and source_dir='src/'
- SageMaker prepares to use a pre-built Scikit-learn container with the specified framework version
2. Code Packaging (on your machine)
- The src/ folder is zipped and uploaded to an internal S3 bucket
- The entry point (train.py) is designated as the script to run inside the container
3. S3 Data Input
- You pass a dictionary like:
- {
  - 'train': 's3://my-bucket/data/train.csv',
  - 'validation': 's3://my-bucket/data/val.csv'
- }
- SageMaker downloads these files to:
    - /opt/ml/input/data/train/train.csv
    - /opt/ml/input/data/validation/val.csv
- SageMaker sets environment variables:
    - SM_CHANNEL_TRAIN
    - SM_CHANNEL_VALIDATION
    - SM_MODEL_DIR
    - SM_OUTPUT_DATA_DIR
- Job Execution (on SageMaker)
    - SageMaker launches the container and runs:
        - python train.py
    - Your train.py script reads from the env vars to:
        - Load data
        - Train model
        - Save model + outputs
- Job Completion
    - Model artifacts are stored in: s3://<your-sagemaker-job-output-bucket>/output/model.tar.gz
    - You can deploy the model or download it

Summary
- train.py	Core ML logic, executed inside container
- source_dir	Folder zipped and uploaded to SageMaker
- run_train.py	Launch job from local VS Code
- .fit({...})	Tells SageMaker where to get data
- Env Vars	Allow train.py to load paths flexibly
- cwd	Base path when resolving relative paths locally

## Further notes on How to Download Files from S3 (Using `boto3`)

When working with SageMaker or local scripts, you may want to download specific files from an S3 bucket to your local machine for inspection, preprocessing, or manual testing.

---

### ✅ Prerequisite

Install `boto3` if not already installed:

```bash
pip install boto3


### Downloading files to local computer from S3
```python
import boto3
import os

# Create an S3 client
s3 = boto3.client('s3')

# Define the S3 location
bucket_name = 'my-bucket'
s3_key = 'data/train/train.csv'  # full path within the bucket

# Define local destination path
local_dir = os.path.join(os.getcwd(), 'downloads')
os.makedirs(local_dir, exist_ok=True)

local_path = os.path.join(local_dir, 'train.csv')

# Download the file
s3.download_file(bucket_name, s3_key, local_path)

# Upload the file 
s3.upload_file(local_path, bucket_name, s3_key)

print(f"File downloaded to: {local_path}")

### What is boto3.client('s3')?
This creates an S3 client object — your Python interface to communicate with the S3 service over the internet. Think of it like this:
- It's your remote control for S3 — the s3 object allows you to send instructions (like "download this file") from your local machine to AWS S3.
- Once you have this object, you can:
    - upload_file() to S3
    - download_file() from S3
    - list_objects() in a bucket
    - delete_object() from S3
    - and more.

#### How does it authenticate?
The boto3.client() uses your credentials from one of the following:
- ~/.aws/credentials file: Created via aws configure on your terminal
- Environment variables	AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY
- IAM role (on SageMaker or EC2)	Automatically handled by AWS — no manual setup
- These credentials tell AWS who you are and whether you’re allowed to access that bucket or file.

#### Function signature (download)
s3.download_file(Bucket, Key, Filename)
- Bucket:	Name of the S3 bucket
- Key:	Full path to the file inside the bucket (including folders)
- Filename:	Local path where the file will be saved

#### Function signature (upload)
s3.upload_file(local_path, bucket_name, s3_key)
- local_path	Path to your file on local machine
- bucket_name	Name of your S3 bucket
- s3_key	Path + name to save as in the bucket
Use this to 
- Save trained models (model.pkl)
- Upload preprocessed data
- Push logs or metrics back to S3

#### Local File Handling Best Practices
If you only specify a filename (e.g. 'train.csv'), the file will be downloaded to your current working directory (os.getcwd()). It's safer to:
- Dynamically build your local path using os.path.join()
- Ensure the destination folder exists using os.makedirs(..., exist_ok=True)
- Always check if the file exists to avoid redundant downloads.
    - if not os.path.exists(local_path):
        - s3.download_file(bucket_name, s3_key, local_path)